# 01 - Dataset: drone landscape footage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matu1003/Makeitalive/blob/main/notebooks/01_dataset.ipynb)

Both methods learn motion from real videos. We use a long compilation of 4K drone footage
(*10 Hours Fantastic Views of Nature*) and turn it into training data:

- **Motion Flow**: pairs of frames `(I_t, I_{t+k})`, the model learns to predict the motion from `I_t` to `I_{t+k}`.
- **SVD + LoRA**: short 14-frame clips, extracted in `03_svd_lora.ipynb` since their sampling is part of each experiment.

A compilation contains many hard cuts between shots, so the key step is to **filter out pairs that span a scene change**.

In [ ]:
import sys
from pathlib import Path

# On Colab, clone the repo and install it; locally, run `uv sync` first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/matu1003/Makeitalive.git
    %cd Makeitalive
    !pip install -q -e .
    REPO_ROOT = Path.cwd()
else:
    REPO_ROOT = Path.cwd().parent

DATA_DIR = REPO_ROOT / "data"
CKPT_DIR = REPO_ROOT / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

import cv2
import numpy as np
import matplotlib.pyplot as plt

from data.download_youtube import download_video
from data.make_dataset_video import extract_pairs_from_video
from data.dataset import LandscapeMotionDataset
from data.video_utils import histogram_distance, scene_change_distance, mse_distance

VIDEO_URL = "https://www.youtube.com/watch?v=AKeUssuu3Is"
VIDEO_PATH = DATA_DIR / "10hourslandscape.mp4"
PAIRS_DIR = DATA_DIR / "dataset_local"

## 1. Download the source video

The video stream is downloaded without audio, in H.264 and at most 720p, so that OpenCV can read it.

In [ ]:
if not VIDEO_PATH.exists():
    download_video(VIDEO_URL, str(VIDEO_PATH), max_height=720)
print(f"Video: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1e6:.0f} MB)")

## 2. Extract image pairs for Motion Flow

One pair is sampled every `interval` seconds. Frame B is taken `gap` frames after frame A, so the motion between
them is small and mostly made of natural movement (water, vegetation, clouds) and slow camera motion.
Frames are resized and center-cropped to 512x512. With `clean=True`, pairs containing a scene change are dropped.

In [ ]:
extract_pairs_from_video(
    video_path=str(VIDEO_PATH),
    output_dir=str(PAIRS_DIR),
    sample_every_n_seconds=5.0,
    frame_gap=4,
    target_size=512,
    max_pairs=500,
    clean=True,
)

## 3. Scene-change filtering

Three distances are compared on every pair:

| Metric | Idea | Scene-change threshold |
|---|---|---|
| `mse_distance` | pixel-wise MSE, very sensitive to any motion | not used for filtering |
| `histogram_distance` | Bhattacharyya distance between color histograms | 0.30 |
| `scene_change_distance` | MAE between 16x16 grayscale thumbnails, blind to small motion | 0.20 |

A pair is rejected if either of the last two exceeds its threshold. To see what the filter protects against,
we compare the kept pairs with *fake* pairs made of two unrelated frames of the dataset.

In [ ]:
dir_A, dir_B = PAIRS_DIR / "img_A", PAIRS_DIR / "img_B"
names = sorted(p.name for p in dir_A.glob("*.jpg"))

def distances(a, b):
    return histogram_distance(a, b), scene_change_distance(a, b)

rng = np.random.default_rng(0)
real, fake = [], []
for name in names:
    a, b = cv2.imread(str(dir_A / name)), cv2.imread(str(dir_B / name))
    real.append(distances(a, b))
    other = cv2.imread(str(dir_B / names[rng.integers(len(names))]))
    fake.append(distances(a, other))
real, fake = np.array(real), np.array(fake)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, i, title, thr in zip(axes, [0, 1], ["Histogram distance", "Thumbnail distance"], [0.30, 0.20]):
    ax.hist(real[:, i], bins=30, alpha=0.7, label="kept pairs (I_t, I_t+k)")
    ax.hist(fake[:, i], bins=30, alpha=0.7, label="unrelated frames")
    ax.axvline(thr, color="k", linestyle="--", label=f"threshold = {thr}")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

The worst kept pairs (highest histogram distance) are the ones to check by eye: they are the most likely false negatives of the filter.

In [ ]:
worst = np.argsort(real[:, 0])[::-1][:4]
fig, axes = plt.subplots(len(worst), 2, figsize=(8, 4 * len(worst)))
for row, idx in zip(np.atleast_2d(axes), worst):
    for ax, d, label in zip(row, [dir_A, dir_B], ["A", "B"]):
        ax.imshow(cv2.cvtColor(cv2.imread(str(d / names[idx])), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{names[idx]} ({label}), hist = {real[idx, 0]:.2f}")
        ax.axis("off")
plt.tight_layout()
plt.show()

## 4. What the model has to learn

`LandscapeMotionDataset` loads the pairs as tensors in `[0, 1]`. The Motion Flow model is trained without any
ground-truth flow, but a classical optical flow estimator (Farneback, OpenCV) shows what kind of motion field
links the two frames.

In [ ]:
dataset = LandscapeMotionDataset(str(PAIRS_DIR))
print(f"{len(dataset)} pairs")

img_A, img_B = dataset[0]
a = (img_A.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
b = (img_B.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

flow = cv2.calcOpticalFlowFarneback(
    cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.cvtColor(b, cv2.COLOR_RGB2GRAY), None,
    pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(a); axes[0].set_title("Frame A (I_t)")
axes[1].imshow(b); axes[1].set_title("Frame B (I_t+k)")
for ax, i, title in zip(axes[2:], [0, 1], ["Flow dx (U)", "Flow dy (V)"]):
    im = ax.imshow(flow[..., i], cmap="coolwarm")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()